# 05 - Feature Engineering y preparación para modelado

Este notebook transforma el dataset auditado en una matriz de modelado reproducible, evitando fuga de información.

**Objetivos**
- Derivar variables útiles a partir de fechas y comportamiento.
- Excluir campos no aptos para el modelo principal.
- Separar train/test antes de imputar o escalar.
- Construir un `ColumnTransformer` reproducible.
- Verificar que el pipeline produzca una matriz sin valores faltantes.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
DATA_PATH = Path('../data/raw/sales_marketing_customer_raw.csv')
df = pd.read_csv(DATA_PATH)
df.shape


## 1. Feature engineering

La fecha de referencia se define como el día posterior a la fecha más reciente observada en el dataset. Esto permite reproducibilidad sin utilizar la fecha del sistema.

Se crean variables derivadas interpretables:
- antigüedad del cliente;
- recencia de compra;
- presencia de cupón;
- engagement relativo de email;
- gasto por visita;
- tickets por visita;
- frecuencia mensual equivalente;
- indicadores de ausencia de datos.


In [ ]:
for col in ['signup_date', 'last_purchase_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

reference_date = max(df['signup_date'].max(), df['last_purchase_date'].max()) + pd.Timedelta(days=1)

df['age_invalid'] = df['age'].notna() & ~df['age'].between(18, 100)
df.loc[df['age_invalid'], 'age'] = np.nan

df['temporal_inconsistency'] = (
    df['signup_date'].notna()
    & df['last_purchase_date'].notna()
    & (df['last_purchase_date'] < df['signup_date'])
)

df['customer_tenure_days'] = (reference_date - df['signup_date']).dt.days
df['recency_days'] = (reference_date - df['last_purchase_date']).dt.days

df['has_coupon'] = df['coupon_code'].notna().astype(int)

df['email_engagement_ratio'] = np.where(
    df['email_open_rate'].fillna(0) > 0,
    df['email_click_rate'] / df['email_open_rate'],
    0.0
)

df['spend_per_visit'] = np.where(
    df['total_visits'].fillna(0) > 0,
    df['total_spent'] / df['total_visits'],
    np.nan
)

df['support_ticket_rate'] = np.where(
    df['total_visits'].fillna(0) > 0,
    df['support_tickets'] / df['total_visits'],
    np.nan
)

df['purchase_frequency_per_month'] = df['last_3_month_purchase_freq'] / 3.0

for col in ['age', 'total_spent', 'gender', 'satisfaction_score']:
    df[f'{col}_missing'] = df[col].isna().astype(int)

print('Fecha de referencia:', reference_date.date())
print('Dimensión luego de feature engineering:', df.shape)


## 2. Variables excluidas del modelo principal

Se excluyen:
- `customer_id`: identificador, no predictor.
- `city`: inconsistencia geográfica detectada en la auditoría.
- fechas crudas: se reemplazan por variables derivadas.
- `coupon_code`: código de alta cardinalidad; se conserva `has_coupon`.
- `lifetime_value`: se excluye del modelo principal porque su definición temporal no está garantizada y podría incorporar información posterior al punto de predicción.
- flags de calidad (`age_invalid`, `temporal_inconsistency`): sirven para auditoría, no para aprender artefactos del dataset.
- `churn`: variable objetivo.


In [ ]:
TARGET = 'churn'

EXCLUDED = [
    'customer_id',
    'city',
    'signup_date',
    'last_purchase_date',
    'coupon_code',
    'lifetime_value',
    'age_invalid',
    'temporal_inconsistency',
    TARGET
]

features = [col for col in df.columns if col not in EXCLUDED]
X = df[features].copy()
y = df[TARGET].astype(int)

categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

print('Variables de entrada:', len(features))
print('Numéricas:', len(numeric_features))
print('Categóricas:', len(categorical_features))
display(pd.DataFrame({'feature': features}))


## 3. Split estratificado antes del preprocessing

La imputación, el escalado y el one-hot encoding se ajustan **solo con el conjunto de entrenamiento**. Así evitamos que información estadística del test influya en el entrenamiento.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train:', X_train.shape, '| churn:', round(y_train.mean(), 4))
print('Test :', X_test.shape, '| churn:', round(y_test.mean(), 4))


## 4. Pipeline de preprocesamiento

- Variables numéricas: imputación por mediana + estandarización.
- Variables categóricas: imputación por moda + one-hot encoding.
- Categorías nuevas en producción se ignoran de forma segura con `handle_unknown='ignore'`.


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print('Train transformado:', X_train_t.shape)
print('Test transformado :', X_test_t.shape)
print('Features finales   :', len(feature_names))
print('NaN train:', np.isnan(X_train_t).sum())
print('NaN test :', np.isnan(X_test_t).sum())


## 5. Controles automáticos

Estos `assert` hacen fallar el notebook si se rompe algún supuesto esencial del pipeline.


In [ ]:
assert X_train.shape[0] + X_test.shape[0] == len(df)
assert y_train.isna().sum() == 0 and y_test.isna().sum() == 0
assert set(y.unique()) <= {0, 1}
assert np.isnan(X_train_t).sum() == 0
assert np.isnan(X_test_t).sum() == 0
assert abs(y_train.mean() - y_test.mean()) < 0.01
assert X_train_t.shape[1] == X_test_t.shape[1]

print('VALIDACIÓN OK')


## Conclusión

La matriz de modelado queda preparada sin fuga de información por preprocessing:

- 80% train / 20% test estratificado.
- Imputación y escalado ajustados solo sobre train.
- Categóricas codificadas dentro del pipeline.
- Variables de calidad o trazabilidad excluidas.
- `lifetime_value` reservado para análisis de negocio y para un experimento secundario, no para el modelo principal.

**Siguiente paso:** construir modelos baseline (`DummyClassifier` y Regresión Logística) y evaluar con métricas adecuadas al desbalance.
